In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 111.1 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.1.10
    Uninstalling hf-xet-1.1.10:
      Successfully uninstalled hf-xet-1.1.10
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled tra

In [2]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 66.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re
from transformers import BitsAndBytesConfig
import torch
from kaggle_secrets import UserSecretsClient
import os

In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Using CPU")

secret_label = "HF_TOKEN"
hf_token = UserSecretsClient().get_secret(secret_label)
print("HF-Token successful.")
os.environ["HF_TOKEN"] = hf_token

Using GPU: Tesla T4
HF-Token successful.


## Load model

In [5]:
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

Loading tokenizer...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

Loading model...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

## Load data

In [6]:
test_df = pd.read_csv('/kaggle/input/datasets/linhtron123/legaldata/test_split.csv') 
test_df.head()

,Dialogue,Manipulative,Primary Manipulator,Manipulation Techniques
0,"Judge: I understand, but my hypothetical assum...",0,NaN,NaN
1,"Judge: Well, but if -- why -- your voluntary d...",0,NaN,NaN
2,Judge: Why did you go?\nPlaintiff: I had nothi...,1,Plaintiff,"deflection, minimization"
3,Judge: Tell me what happened.\nPlaintiff: Okay...,1,Defendant,"gaslighting, deflection, emotional appeal, pla..."
4,Judge: Is your position— is there any daylight...,0,NaN,NaN


In [7]:
if 'Dialogue' in test_df.columns:
    dialogue_df = test_df[['Dialogue']].copy()

## System Prompts

In [8]:
system_prompt_plaintiff = (
    "You are reading a transcript from a courtroom conversation.\n\n"
    "Step 1: Carefully read the dialogue.\n"
    "Step 2: Think step-by-step about what the Plaintiff's statements suggest.\n"
    "Step 3: Reason about the Plaintiff's goal or motive behind their words.\n"
    "Step 4: Summarize the Plaintiff's intent in three complete sentences maximum. "
    "Ensure your response is complete and properly punctuated.\n\n"
    "Now analyze the given dialogue.\n"
)

system_prompt_defendant = (
    "You are reading a transcript from a courtroom conversation.\n\n"
    "Step 1: Carefully read the dialogue.\n"
    "Step 2: Think step-by-step about what the Defendant's statements suggest.\n"
    "Step 3: Reason about the Defendant's goal or motive behind their words.\n"
    "Step 4: Summarize the Defendant's intent in three complete sentences maximum. "
    "Ensure your response is complete and properly punctuated.\n\n"
    "Now analyze the given dialogue.\n"
)

## Format output

In [9]:
def extract_and_format_response(text):
    if "Step 4:" in text:
        response_text = text.split("Step 4:")[-1].strip()
    else:
        response_text = text.strip()
    sentences = re.split(r'(?<=[.!?]) +', response_text)
    complete_sentences = [s for s in sentences if s and s[-1] in ['.', '!', '?']]
    limited_sentences = complete_sentences[:3]
    formatted_response = ' '.join(limited_sentences)
    if not formatted_response:
        if response_text and response_text[-1] not in ['.', '!', '?']:
            response_text += '.'
        return response_text[:500] 
    
    return formatted_response

In [10]:
def generate_text(prompt, max_tokens=500): 
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_length = inputs.input_ids.shape[1]
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,  
            pad_token_id=tokenizer.eos_token_id
        )
    
    new_tokens = output[0, input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [11]:
def analyze_plaintiff(dialogue):
    """Analyze plaintiff's intent with robust error handling"""
    messages = [
        {"role": "user", "content": f"{system_prompt_plaintiff}Dialogue:\n{dialogue}\n\nProvide your analysis:"}
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
 
    generated_text = generate_text(input_text, max_tokens=500)

    result = extract_and_format_response(generated_text)
    
    return result
        
def analyze_defendant(dialogue):
    """Analyze defendant's intent with robust error handling"""
        
    messages = [
        {"role": "user", "content": f"{system_prompt_defendant}Dialogue:\n{dialogue}\n\nProvide your analysis:"}
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    generated_text = generate_text(input_text, max_tokens=500)

    result = extract_and_format_response(generated_text)
    
    return result
    

In [12]:
dialogue_df['PLAINTIFF'] = dialogue_df['Dialogue'].apply(analyze_plaintiff)
dialogue_df['DEFENDANT'] = dialogue_df['Dialogue'].apply(analyze_defendant)

dialogue_df.to_csv('test_intent.csv', index=False)
print(f"Saved results")

Saved results
